In [ ]:
cd ../..

## Feature Ideas

- `week_of_year`
- `month_of_year`
- `day_of_month`
- `day_of_week`
- `department_no`
- `section_no`
- `perceived_colour_master_id`
- For each 1 week window in the previous 5 weeks:
    - `num_sales`
    - `sold_amount`
    - `avg_price`
    - `num_customers`
    - Averaged across buyers in that week:
        - `avg_buyer_age`
        - `avg_monthly_purchase_amount`
        - `avg_monthly_purchase_count`
        - `avg_weeks_since_last_purchase`

In [ ]:
import duckdb
from torch_frame.utils import infer_df_stype

import utils

conn = duckdb.connect('hm/hm.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

In [ ]:
with open('hm/item-sales/feats.sql', 'r') as f:
    # run once with train_labels and once with val_labels
    template = f.read()

# create train, val and test features
# takes 1 - 5 mins
for s in ['train', 'val', 'test']:
    print(f'Creating {s} table')
    query = utils.render_jinja_sql(template, dict(set=s, subsample=0))
    conn.sql(query)
    print(f'{s} table created')

In [ ]:
utils.validate_feature_tables('item_sales', conn)

In [ ]:
%%sql train_df <<
from item_sales_train_feats

In [ ]:
infer_df_stype(train_df)

In [ ]:
utils.feature_summary_df(train_df.sample(20_000), 'sales', classification=False)

## Feature Importances

In [ ]:
import shap
from torch_frame import TaskType, stype
from torch_frame.data import Dataset
from torch_frame.gbdt import LightGBM

from inferred_stypes import task_to_stypes
from train_gbdt import TASK_PARAMS

TASK = 'rel-hm-item-sales'

task_params = TASK_PARAMS[TASK]

In [ ]:
%%sql val_df <<
select * from item_sales_val_feats;

In [ ]:
val_tf = Dataset(
    val_df,
    col_to_stype=task_to_stypes[TASK],
    target_col=task_params['target_col'],
).materialize().tensor_frame

In [ ]:
gbdt = LightGBM(task_type=task_params['task_type'])
gbdt.load(f'models/{TASK}_lgbm.json')
pred = gbdt.predict(tf_test=val_tf).numpy()

In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(gbdt.model)

val_arr, _, _ = gbdt._to_lightgbm_input(val_tf[:10_000])
shap_values = explainer.shap_values(val_arr, pred[:10_000])

# TODO verify
feat_names = val_tf.col_names_dict.get(stype.categorical, []) + val_tf.col_names_dict[stype.numerical]

shap.summary_plot(shap_values, val_arr, plot_type='violin', max_display=30, feature_names=feat_names)

In [ ]:
conn.close()